# Function calling SFT: Qwen2.5 3B

Supervised fine tuning (LoRA, Unsloth) of Qwen2.5 3B Instruct on 10,000 examples from
[Salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k),
evaluated on 200 held out examples. See the README for results.

> Runs on a free Colab T4. Checkpoints are pushed to Hugging Face during training.
> Long per example debug prints were removed; the training loss and evaluation tables are kept.


#### Install Dependencies and Setup Environment

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Clear pip cache to ensure clean install
    !pip cache purge
    # Install PyTorch with CUDA 12.4
    !pip install torch==2.6.0+cu124 torchvision==0.19.0+cu124 torchaudio==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124 --no-cache-dir
    # Install dependencies
    !pip install --no-deps bitsandbytes accelerate peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    # Install latest xformers
    !pip install xformers --no-deps --force-reinstall --no-cache-dir
    # Install unsloth last
    !pip install --no-deps unsloth
    # Verify xformers and CUDA
    !python -c "try: import xformers; print(f'xformers version: {xformers.__version__}'); import torch; print(f'CUDA available: {torch.cuda.is_available()}'); except Exception as e: print(f'Error: {str(e)}')"

 Login to Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Setup Repository Paths & Metadata

In [ ]:
%%capture
from huggingface_hub import notebook_login, HfApi, create_repo
import pathlib, json, os, shutil, time

PROJECT_DIR= "Qwen2.5-3B-RAG-API-Finetuned-Models"
BASE_REPO= "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
api= HfApi()
USERNAME = api.whoami()["name"]

# where the base LoRA training checkpoints will land locally
LOCAL_OUT = pathlib.Path("outputs")
LOCAL_OUT.mkdir(exist_ok=True)

# record the repos we create so we can loop over them later
REPO_REGISTRY = pathlib.Path("checkpoint_repos.json")
if not REPO_REGISTRY.exists():
    REPO_REGISTRY.write_text("[]")


 Load Base Model with Unsloth

In [ ]:
# ---- Model loading

from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

#Specify model name
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

#Add LoRA adapters:

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

 #### Load and Preprocess Dataset, Format Data into ChatML Style & Preview Formatted Data

In [ ]:
# ─── DATASET LOADING, FILTERING & CLEAN FORMATTING ─────────────────────────────

import os, json
from datasets import load_dataset
from transformers import AutoTokenizer
from pprint import pprint

# 1) Constants
CACHE_DIR         = os.path.join(os.path.expanduser("~"), ".cache", "huggingface", "datasets")
BASE_MODEL        = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
SYSTEM_PROMPT     = (
    "You are an assistant skilled at generating accurate API calls based on natural "
    "language queries. Given a user query and a list of available tools, output a structured "
    "API call with the correct endpoint and parameters in JSON format."
)
MAX_PROMPT_TOKENS = 1024
TRAIN_SIZE        = 10000
TEST_SIZE         = 500

# 2) Load raw “train” split
raw = load_dataset("Salesforce/xlam-function-calling-60k", cache_dir=CACHE_DIR)["train"]
print("Raw examples:", len(raw), "columns:", raw.column_names)

# 3) Prepare tokenizer (we only need it for token‐counts here)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# 4) Helper: turn one example → {prompt: [system,user], answer: str}
def format_tool_call_sample(example):
    # parse tools JSON
    tools = json.loads(example["tools"]) if isinstance(example["tools"], str) else example["tools"]
    tools_str = "\n".join(
        f"- {t['name']}: {t['description']} | Params: {', '.join(t.get('parameters',{}).keys())}"
        for t in tools
    )
    user_msg = f"Query: {example['query']}\n\nAvailable Tools:\n{tools_str}"
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}
        ],
        "answer": example["answers"]
    }

# 5) Length‐filter: drop any whose ChatML would exceed MAX_PROMPT_TOKENS:
def too_long(example):
    formatted = format_tool_call_sample(example)
    full_chat = formatted["prompt"] + [{"role": "assistant", "content": formatted["answer"]}]
    chatml_text = tokenizer.apply_chat_template(full_chat, tokenize=False)
    return len(tokenizer(chatml_text).input_ids) > MAX_PROMPT_TOKENS


filtered = raw.filter(lambda ex: not too_long(ex), num_proc=4)
print("After filtering:", len(filtered), "dropped", len(raw) - len(filtered))

# 6) Split 10k/500
total = len(filtered)
if total < TRAIN_SIZE + TEST_SIZE:
    TRAIN_SIZE = total * 9 // 10
    TEST_SIZE  = total - TRAIN_SIZE

train_raw = filtered.select(range(TRAIN_SIZE))
test_raw  = filtered.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))

# 7) Keep only the three columns we care about
keep = ["query","tools","answers"]
train_raw = train_raw.remove_columns([c for c in train_raw.column_names if c not in keep])
test_raw  = test_raw.remove_columns([c for c in test_raw.column_names  if c not in keep])

# 8) Save raw subsets
train_raw.to_json("train_subset_raw.json", orient="records", lines=True)
test_raw.to_json("test_subset_raw.json",  orient="records", lines=True)

# 9) Map to clean {prompt:list-of-dicts, answer:str}
train_ds = train_raw.map(
    format_tool_call_sample,
    batched=False,
    remove_columns=train_raw.column_names
)
test_ds  = test_raw.map(
    format_tool_call_sample,
    batched=False,
    remove_columns=test_raw.column_names
)

train_dataset_chatml = train_ds.map(
    lambda ex: {
        "text": tokenizer.apply_chat_template(
            ex["prompt"], tokenize=False, add_generation_prompt=True
        ) + ex["answer"].strip()
    },
    batched=False
)

test_dataset_chatml = test_ds.map(
    lambda ex: {
        "text": tokenizer.apply_chat_template(
            ex["prompt"], tokenize=False, add_generation_prompt=True
        ) + ex["answer"].strip()
    },
    batched=False
)

# 10) Save formatted + Preview
train_ds.to_json("train_dataset_chatml.json", orient="records", lines=True)
test_ds .to_json("test_dataset_chatml.json",  orient="records", lines=True)

print(f"Train size: {len(train_ds)}, Test size: {len(test_ds)}\n")
print("===== Sample train[0] =====")
pprint(train_ds[0])
print("\n===== Sample test[0] =====")
pprint(test_ds[0])


In [ ]:
print(f"Training set size: {len(train_ds)}")
print(f"Testing set size: {len(test_ds)}")

#### Format Data into ChatML Style

In [ ]:
def build_chatml_prompt_only(example, system_prompt=SYSTEM_PROMPT):
    # 1) build the pure system+user conversation
    tools = json.loads(example["tools"]) if isinstance(example["tools"], str) else example["tools"]
    tools_str = "\n".join([
        f"- {tool['name']}: {tool['description']} | Params: {', '.join(tool.get('parameters', {}).keys())}"
        for tool in tools
    ])
    user_content = f"Query: {example['query']}\n\nAvailable Tools:\n{tools_str}"
    convo = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": ""},           # empty assistant slot
    ]

    # 2) render ChatML and print it
    prompt_text = tokenizer.apply_chat_template(
        convo,
        tokenize=False,
        add_generation_prompt=True
    )
    # make sure we never injected the ground‐truth 'answer'
    assert example.get("answers", "") not in prompt_text, "👀 Found answer text in prompt!"
    return prompt_text



###Define Training Arguments & Initialize Trainer

In [ ]:
# fine‑tuning configuration + Trainer (prints every 10th step + saves step‑10 only)

import json
import pathlib
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
from unsloth import FastLanguageModel, is_bfloat16_supported
from huggingface_hub import HfApi, create_repo, Repository

# ---------- project constants --------------------------------------
LOCAL_OUT   = pathlib.Path("outputs")
LOCAL_OUT.mkdir(exist_ok=True)
CHECKPOINT_INTERVAL = 100

# ---------- helper: quick demo generation on test_subset[0] --------
def run_live_inference(model, tokenizer, step=None):
    # pick a single example from your test split
    example = test_raw[1]

    # show which example we're using
    print(f"\n📥 [run_live_inference] Step={step or 'N/A'}  Query={example['query']!r}\n")

    # 1) build ChatML prompt
    prompt_text = build_chatml_prompt_only(example)

    # 2) debug: show the full prompt so you can eyeball it
    print("--- DEBUG: Prompt Text ---")
    print(prompt_text)

    # 3) sanity check: ensure the model never sees the gold answer
    assert example.get("answers", "") not in prompt_text, (
        "👀 Found ground-truth answers in the prompt!"
    )

    # 4) tokenize & move to device
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # 5) generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs.get("attention_mask"),
            max_new_tokens = 1024,
            use_cache      = True,
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.pad_token_id,
        )

    # 6) strip off the prompt tokens to just get the new generation
    prompt_len      = inputs["input_ids"].shape[1]
    gen_ids         = outputs[0, prompt_len:]

    # 7) decode & print only the generated JSON
    generated_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    print(f"\n🧪 Model response @ step {step}:\n{generated_text}\n{'─'*60}\n")

    return generated_text


# ---------- callback ------------------------------------------------
class CheckpointAndPreview(TrainerCallback):
    def __init__(self, tokenizer, checkpoint_interval=100):
        self.tokenizer = tokenizer
        self.checkpoint_interval = checkpoint_interval
        create_repo(PROJECT_DIR, repo_type="model", exist_ok=True)
        self.repo_dir = LOCAL_OUT / PROJECT_DIR
        self.repo_dir.mkdir(exist_ok=True)
        self.repo = Repository(
            local_dir     = str(self.repo_dir),
            clone_from    = f"{USERNAME}/{PROJECT_DIR}",
            use_auth_token= True,
            repo_type     = "model"
        )

    def on_log(self, args, state, control, **kwargs):
        if state.global_step % self.checkpoint_interval == 0:
            run_live_inference(kwargs["model"], self.tokenizer, state.global_step)

    def on_save(self, args, state, control, **kwargs):
        if state.global_step % self.checkpoint_interval != 0:
            return

        step = state.global_step
        subdir = self.repo_dir / f"step-{step}"
        subdir.mkdir(exist_ok=True)

        kwargs["model"].save_pretrained(subdir)
        self.repo.push_to_hub(commit_message=f"Add checkpoint step-{step}")
        print(f"💾 Pushed step-{step} into repo {USERNAME}/{PROJECT_DIR}")
        #run_live_inference(kwargs["model"], self.tokenizer, step)




# ---------- TrainingArguments --------------------------------------
training_args = TrainingArguments(
    output_dir                   = LOCAL_OUT,
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,
    warmup_steps                 = 5,
    #max_steps                    = 1250,  # or any number you want
    num_train_epochs             = 1,
    learning_rate                = 2e-4,
    fp16                         = not is_bfloat16_supported(),
    bf16                         =     is_bfloat16_supported(),
    logging_steps                = CHECKPOINT_INTERVAL,
    save_steps                   = CHECKPOINT_INTERVAL,
    save_total_limit             = 1,
    report_to                    = "none",
    push_to_hub                  = False,  # still handled manually in your callback
)


# ---------- build & launch trainer ---------------------------------
FastLanguageModel.for_inference(model)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset_chatml,
    dataset_text_field = "text",
    max_seq_length     = 1024,
    args               = training_args,
    callbacks          = [CheckpointAndPreview(tokenizer, CHECKPOINT_INTERVAL)],
)


### Start Fine-tuning

In [ ]:
# --- Run training ---
trainer.train(resume_from_checkpoint=False)
#trainer.train(resume_from_checkpoint=True)

# --- Optionally push final model to HF:
trainer.push_to_hub()

Step,Training Loss
100,0.766400
200,0.692700
300,0.640000
400,0.600200
500,0.576600
600,0.534700
700,0.523200
800,0.517700
900,0.483800
1000,0.458200


### Evaluation Reward Functions (for Fine-tuning Performance)

In [ ]:
#UPDATED 18/4/25
# Cell 1: Reward functions for Salesforce function calling dataset

from collections import Counter
import json, re

# ---------- 1. Format Reward ----------
def format_reward_func(completions, **kwargs):
    """
    ✅ Reward for producing syntactically correct structured output.

    Checks whether the model's output is:
    - Valid JSON
    - A list of objects
    - Each object has a 'name' and 'arguments' key

    🎯 Rewards:
    - +0.5 for structurally valid outputs
    - -0.2 for outputs that are not parseable JSON (strong penalty)

    🧾 Output Range: [-0.2, 0.5]
    """
    rewards = []
    for completion in completions:
        try:
            parsed = json.loads(completion[0]["content"])
            if isinstance(parsed, list) and all(
                isinstance(t, dict) and "name" in t and "arguments" in t
                for t in parsed
            ):
                rewards.append(0.5)
            else:
                rewards.append(0.0)
        except json.JSONDecodeError:
            rewards.append(-0.2)
    return rewards

# ---------- 2. Tool Name Reward ----------
def tool_name_reward_func(completions, answer, **kwargs):
    """
    ✅ Reward for selecting correct tool/function names in model completions.

    🎯 Output Range: [0.0, 1.0]
    """
    rewards = []
    for completion, gold_str in zip(completions, answer):
        content = completion[0]["content"]
        try:
            pred = json.loads(content)
            gold = json.loads(gold_str)
            pred_tools = [t.get("name") for t in pred if isinstance(t, dict)]
            gold_tools = [t["name"] for t in gold]

            pred_counts = Counter(pred_tools)
            gold_counts = Counter(gold_tools)
            matched = sum(min(pred_counts[t], gold_counts[t]) for t in gold_counts)
            base_score = matched / len(gold_tools) if gold_tools else 0.0

            hallucinated = [t for t in pred_tools if t not in gold_tools]
            penalty = 0.2 * len(hallucinated)
            final = max(0.0, base_score - penalty)
            rewards.append(final)
        except Exception:
            gold = json.loads(gold_str)
            gold_tools = list({t["name"] for t in gold})
            found = [
                tool for tool in gold_tools
                if re.search(rf"\b{re.escape(tool.lower())}\b", content.lower())
            ]
            rewards.append(len(found) / len(gold_tools) if gold_tools else 0.0)
    return rewards

# ---------- 3. Parameter Key Reward ----------
def param_key_reward_func(completions, answer, **kwargs):
    """
    ✅ Reward for selecting correct parameter keys in tool arguments.

    🎯 Output Range: [0.0, 1.0]
    """
    rewards = []
    for completion, gold_str in zip(completions, answer):
        content = completion[0]["content"]
        try:
            pred = json.loads(content)
            gold = json.loads(gold_str)
            pred_keys = [
                key for t in pred if isinstance(t, dict)
                for key in t.get("arguments", {}).keys()
            ]
            gold_keys = [
                key for g in gold
                for key in g["arguments"].keys()
            ]

            pred_counts = Counter(pred_keys)
            gold_counts = Counter(gold_keys)
            matched = sum(min(pred_counts[k], gold_counts[k]) for k in gold_counts)
            base_score = matched / len(gold_keys) if gold_keys else 0.0

            hallucinated = [k for k in pred_keys if k not in gold_keys]
            penalty = 0.1 * len(hallucinated)
            final = max(0.0, base_score - penalty)
            rewards.append(final)
        except Exception:
            gold = json.loads(gold_str)
            gold_keys = list({k for g in gold for k in g["arguments"].keys()})
            found = [
                k for k in gold_keys
                if re.search(rf"\b{re.escape(k.lower())}\b", content.lower())
            ]
            rewards.append(len(found) / len(gold_keys) if gold_keys else 0.0)
    return rewards

# ---------- 4. Parameter Value Reward ----------
def param_value_reward_func(completions, answer, **kwargs):
    """
    ✅ Reward for predicting correct parameter values.

    🎯 Output Range: [0.0, 1.0]
    """
    rewards = []
    for completion, gold_str in zip(completions, answer):
        try:
            pred = json.loads(completion[0]["content"])
            gold = json.loads(gold_str)
            pred_vals = [
                str(v).strip().lower()
                for t in pred if isinstance(t, dict)
                for v in t.get("arguments", {}).values()
            ]
            gold_vals = [
                str(v).strip().lower()
                for g in gold
                for v in g["arguments"].values()
            ]
            correct = len(set(pred_vals) & set(gold_vals))
            total = len(set(gold_vals))
            rewards.append(correct / total if total > 0 else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

# ---------- 5. Aggregated Reward ----------
def aggregated_reward_func(completions, answer, **kwargs):
    """
    ✅ Combines all the other reward functions into a single scalar score.
    🧾 Output Range: [-0.2, ~3.0]
    """
    f_scores = format_reward_func(completions)
    t_scores = tool_name_reward_func(completions, answer)
    k_scores = param_key_reward_func(completions, answer)
    v_scores = param_value_reward_func(completions, answer)
    return [f + t + 0.5*k + v for f, t, k, v in zip(f_scores, t_scores, k_scores, v_scores)]


### Evaluation Setup for Checkpoint Inference

In [ ]:
# ─── Inference & Evaluation Pipeline with local adapters ────────────────────────────

import os, json, torch, textwrap, pandas as pd
from huggingface_hub import HfApi, snapshot_download
from unsloth import FastLanguageModel, get_chat_template
from peft import PeftModel
from datasets import Dataset

# User settings
TEST_JSON = "test_subset_raw.json"
NUM_EX    = 200
FULL_REPO = f"{USERNAME}/{PROJECT_DIR}"

print(f"🔍 Downloading main repo '{FULL_REPO}'...")
local_repo = snapshot_download(FULL_REPO, repo_type="model")
print("✅ Local repo path:", local_repo)

# Discover adapter subfolders (step-*)
adapter_steps = sorted(
    [d for d in os.listdir(local_repo)
     if d.startswith("step-") and 700 <= int(d.split("-")[1]) <= 1000],
    key=lambda x: int(x.split("-")[1])
)
print("Found adapter snapshots:", adapter_steps)

# Load base model & tokenizer once
base_model, base_tok = FastLanguageModel.from_pretrained(
    BASE_REPO,
    load_in_4bit=True,
    max_seq_length=4096,
    device_map="auto"
)
FastLanguageModel.for_inference(base_model)
tok = get_chat_template(
    base_tok,
    chat_template="chatml",
    mapping={"role":"from","content":"value","user":"human","assistant":"gpt"},
    map_eos_token=True,
)

# Prepare data & prompts
ds = Dataset.from_json(TEST_JSON).select(range(NUM_EX))
SYSTEM = (
    "You are an assistant skilled at generating accurate API calls based "
    "on natural-language queries. Given a user query and a list of tools, "
    "output a structured API call in JSON format."
)
def run_and_collect(model):
    comps, golds = [], []
    for i, ex in enumerate(ds):
        # 1) build the full ChatML prompt
        prompt_text = build_chatml_prompt_only(ex)

        # 2) tokenize that prompt (ChatTemplate wrapper → tensors)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        prompt_len = inputs["input_ids"].shape[1]

        # 3) generate
        out = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs.get("attention_mask"),
            max_new_tokens = 1024,
            pad_token_id   = tok.pad_token_id,
            eos_token_id   = tok.eos_token_id,
        )[0]

        # 4) strip off prompt tokens & decode
        gen = tokenizer.decode(out[prompt_len:], skip_special_tokens=True)


        comps.append([{"content": gen}])
        golds.append(ex["answers"])

    return comps, golds

# Evaluate base + each adapter
results = []
def evaluate_model(label, model):
    comps, golds = run_and_collect(model)
    n = len(comps)
    return {
        'checkpoint': label,
        'format': sum(format_reward_func(comps)) / n,
        'tool':   sum(tool_name_reward_func(comps, golds)) / n,
        'key':    sum(param_key_reward_func(comps, golds)) / n,
        'value':  sum(param_value_reward_func(comps, golds)) / n,
    }

# print("\n===== BASE MODEL =====")
# results.append(evaluate_model('base', base_model))

for step in adapter_steps:
    print(f"\n===== {step.upper()} =====")
    adapter_dir = os.path.join(local_repo, step)
    ft = PeftModel.from_pretrained(base_model, adapter_dir, device_map="auto")
    FastLanguageModel.for_inference(ft)
    results.append(evaluate_model(step, ft))
    del ft; torch.cuda.empty_cache()

# Summary table
for r in results:
    r['agg'] = r['format'] + r['tool'] + 0.5 * r['key'] + r['value']

df = pd.DataFrame(results)
print("\n=== Metrics over first", NUM_EX, "examples ===")
print(df.to_markdown(index=False))


=== Metrics over first 200 examples ===
| checkpoint   |   format |   tool |      key |    value |     agg |
|:-------------|---------:|-------:|---------:|---------:|--------:|
| step-700     |      0.5 |  0.986 | 0.945458 | 0.836542 |
| step-800     |   0.4965 | 0.9895 | 0.947708 | 0.835542 |
| step-900     |   0.4965 | 0.9895 | 0.958292 | 0.847292 |
| step-1000    |      0.5 |  0.988 |  0.95825 | 0.855292 |
(agg column omitted; see README)
